In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


CUDA available: True
GPU: Tesla T4


In [4]:
from google.colab import drive
drive.mount('/content/drive')


NotImplementedError: Mounting drive is unsupported in this environment. Use PyDrive2 instead. See examples at https://colab.research.google.com/notebooks/io.ipynb#scrollTo=7taylj9wpsA2.

## Sarvam ASR Assignment: Diarization Benchmarking & Enhancement
---

**Deadline: Saturday, 22 August 2026, EOD**

You are given a CSV containing 100 YouTube videos with ground-truth diarization labels and reference transcripts. Your task is to parse these labels, benchmark existing diarization and ASR systems on them, and then **improve** the output.

### Input

`youtube_segments.csv` — one row per segment. The csv is present at https://drive.google.com/file/d/1Ijs1IWypIY2GAjpNUKV2XNZY6o7dFvSm/view?usp=sharing

Columns:

- `video_id` — YouTube video id
- `youtube_link` — source URL
- `start_sec`, `end_sec` — the window to extract. Use **only** `[start_sec, end_sec]`.
- `diarization_segments` — **ground-truth diarization**, formatted as `Speaker A [start-end] | Speaker B [start-end] | ...`
- `asr_segments` — **reference transcript** for the same segment, formatted as `[start-end] text | [start-end] text | ...`

`diarization_segments` and `asr_segments` share the **same segment boundaries in the same order** — the *n*-th `asr_segments` entry is the transcript of the *n*-th `diarization_segments` turn — so you can join speaker to text by index or by time. All timestamps are **relative to `start_sec`** (0 = start of the extracted clip).

If for any reason the ground-truth labels look incorrect, hand-label a subset and call out the limitation.

---

### Step 1 - Audio (or Video) Extraction

- Download the **audio** track from each YouTube link (yt-dlp / ffmpeg).
- Trim to `[start_sec, end_sec]`.
- Store as 16 kHz mono WAV.
- Optionally download the **video** track for the same window as well, if you plan to use visual cues in Step 4.

### Step 2 - Baseline Diarization Benchmarking

Run the extracted segments through top open-source diarization models.

Report standard metrics:

- **DER**
- **JER**
- Speaker-count accuracy

**Important:** Do NOT ignore overlapping speech regions when computing metrics. Overlap must be scored.

### Step 3 - ASR on Diarized Segments

Run ASR on the diarized speaker segments to produce speaker-attributed transcripts. Benchmark more than one STT system (e.g. Sarvam Saaras, Whisper, etc.) — the reference transcript in `asr_segments` is what you score against, not a substitute for running ASR yourself. Think about strategies for handling overlapping speech regions during ASR - how do you transcribe segments where multiple speakers are active simultaneously?

Also report:

- **cpWER**
- **WDER**

### Step 4 - Improving the Output

Take the best ASR + diarization combination you found while benchmarking, and build a pipeline on top of it that improves the output. Quantify the improvement over the same metrics as before. It is up to you whether you target diarization, ASR, or both.

**The ground truth is never an input to your pipeline.** `diarization_segments` and `asr_segments` are only for computing the final scores.

You are free to choose the strategy. Some directions:

- **Semantic / LLM-based correction** — feed the speaker-attributed text + timestamps to an LLM to detect and correct boundary errors, speaker confusion, and turn-merging issues. Any LLM is acceptable: paid APIs (GPT, Claude, Gemini, etc.) or open-source models. Prompting strategy, structured-output schema, chain-of-thought, and multi-pass refinement are yours to design.
- **Acoustic and visual cues** — use the video alongside the audio (e.g. active-speaker detection, face tracks, lip motion) to refine speaker boundaries and identities.
- Any combination of the above, or anything else you can justify.

**Hint:** Think about how transcript content can reveal diarization mistakes - e.g. unnaturally short segments, repeated/stuttered text across a speaker boundary suggesting a false split, or incoherent speaker transitions that indicate speaker confusion.

Also include:
- The papers / ideas your design draws from
- Multilingual / Indic-specific observations and failure modes

---

### Deliverables

Mail a Google Drive folder link (with all of the following):

1. A reproducible Colab notebook with the full pipeline end-to-end.
2. A results table: baseline vs. improved DER / JER / cpWER / WDER per model per video.
3. A short writeup (1–2 pages) covering approach, design choices, what worked, what didn't, failure cases, and references.

---

### Evaluation

- Correctness of pipeline and metrics
- Magnitude and consistency of improvement over baseline
- Depth of analysis and engineering judgement
- Clarity of code and writeup
- Breadth of techniques explored (bonus)


    

---

# Step 0 — Dataset: structure and observations

Everything below was measured from the real `youtube_segments.csv` (100 rows,
9,942 segments) before any code was written. These facts drive the design of
every later step, so they are recorded here rather than in a side document.

### Shape and keys

| Fact | Value | Why it matters |
|---|---|---|
| Rows × columns | 100 × 6, no nulls | — |
| Column order in file | `video_id, start_sec, end_sec, youtube_link, diarization_segments, asr_segments` | **Differs from the brief.** Index by name, never by position. |
| Unique `video_id` | **99**, not 100 | `RL2fhIEEbZg` appears **twice**, with the disjoint windows `0–65` and `66–1034`. The primary key is `(video_id, start_sec, end_sec)`, so every filename and cache key uses a composite `clip_id`. |
| `youtube_link` | always `https://www.youtube.com/watch?v={video_id}` | Redundant; `video_id` is the reliable field. |
| `video_id` charset | all match `^[A-Za-z0-9_-]{11}$`, 29 contain `-`/`_`, none *start* with one | No CLI arg-parsing hazard, but IDs are still passed as argv lists, never shell strings. |
| `start_sec` / `end_sec` | floats, always integral | start 0–677 (26 rows start at 0), end 60–2005. |

### Volume

Clip length is bimodal: min 50 s, median 325 s, max 1822 s — 38 rows are ≤120 s
while 27 rows exceed 600 s. **Total 12.28 h**, which is ≈1.41 GB as 16 kHz mono
s16 WAV. The full source videos total 29.3 h, so fetching only the requested
window saves ~58% of the bytes. No row's `end_sec` exceeds its source video's
duration, so every window is extractable in principle.

### Availability

Checked live. **99 of 99 videos are reachable except one**: `GUVrL5ltiP4` is
deleted (`Video unavailable`), so the ceiling for Step 1 is **99/100 rows**.

> `2HGP34TNvjg` returns HTTP 401 from YouTube's oEmbed endpoint, which looks
> like a dead video but only means *embedding is disabled* — yt-dlp downloads it
> normally. Availability is judged by an actual fetch, never by an oEmbed probe.

### Label format

Both label columns are `|`-separated, and splitting on `|` then anchoring a
timestamp regex at each piece parses **100% of 9,942 segments** with zero
failures. Two details make the naive approach wrong:

* **82 transcript segments contain literal `[` or `]`** — square-bracket
  code-switch glosses such as `டவுட் [doubt]`. A global `findall` over
  `\[...\]` mis-parses the file, so the regex is anchored at the start of each
  piece.
* `diarization_segments[n]` and `asr_segments[n]` carry **byte-identical
  timestamps for every single segment** (0 mismatches), so index-join and
  time-join are equivalent. Segments are already sorted by start time.

Speaker labels are `Speaker A`–`Speaker H`, always contiguous from A. Speakers
per clip — the reference for Step 2's speaker-count accuracy:

| Speakers | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|
| Clips | 26 | 29 | 28 | 9 | 4 | 2 | 2 |

### Timing quirks that change how metrics must be computed

* **Ground truth runs past the clip in 85 / 100 rows** — median **+1.43 s**,
  max **+6.41 s** (`Iqhnt8ENm3U`), uncorrelated with clip length (r = −0.11), so
  it is a systematic annotation-window offset rather than drift. Three rows go
  the other way, with long unannotated tails: `Cku_X_SL7qU` **−90.3 s**,
  `OxYCBQKZ3iY` −12.4 s, `CO_8ppdzq9U` −6.1 s.
  ⇒ Step 1 publishes exactly `[start_sec, end_sec]`, and Step 2 must **crop the
  reference to the UEM** rather than extend the audio.
* **Overlap is pervasive and must be scored** (the brief forbids ignoring it):
  **7.16% of corpus time has ≥2 distinct speakers active**, and only **9 of 100
  clips** have no overlap at all. `QuA_B6IZ6Ls` is 35% overlapped.
  *(Summing pairwise overlaps instead gives 8.49% — that double-counts 3-way
  overlap, so the profile uses a sweep-line over elementary intervals.)*
* **Turns are utterance-level, not turn-level**: 22.4% of adjacent segment pairs
  share a speaker label, so a merged/turn-level view must be compared too.
* **20% of segments are under 0.5 s** (1,953 of them; 3,473 are under 1 s),
  median segment 1.89 s. These are backchannels, and most diarizers miss them.

### Two corrupt ground-truth segments

Dropped by the parser with a flag, never silently:

| Clip | Segment | Problem | Text |
|---|---|---|---|
| `QuA_B6IZ6Ls` | 93 | `Speaker C [939.09-155.91]` — **end < start** (−783 s); neighbours sit at 931.5 and 947.0, so the intended end was ~945.9 | `<unintelligible>` |
| `6ZeRgvDHwcI` | 0 | `Speaker A [1.61-1.61]` — **zero duration** | `<noise>` |

### Content: multilingual and heavily code-switched

The corpus spans **nine Indic scripts**, so Step 3 cannot use a single fixed ASR
language code — the per-clip `lang_hint` column carries it forward:

| Script | Deva | Gujarati | Telugu | Tamil | Kannada | Odia | Bengali | Gurmukhi | Malayalam |
|---|---|---|---|---|---|---|---|---|---|
| Clips | 25 | 12 | 12 | 10 | 9 | 9 | 8 | 8 | 7 |

Devanagari covers both Hindi and Marathi videos and is labelled `hi_or_mr`
rather than guessed. **26.1% of all transcript characters are Latin** (median
video 27%, heaviest 48%) — code-switching is the norm, not an edge case.

**Non-speech tags**: `<unintelligible>` 264, `<noise>` 166, `<laughter>` 148,
`<vocalization>` 81, `<background_speech>` 12, `<uhhh>` 1. **404 segments (4.1%)
are tag-only or empty** — real diarization turns, but they must be excluded from
WER scoring.

**The code-switch gloss convention is messier than a single rule can handle.**
22,547 glosses across 4,624 segments (46%). A Step 3 normalizer must survive all
of these, or cpWER is measuring the convention rather than the ASR:

| Variant | Example | Count |
|---|---|---|
| No space before paren | `कॉफी(coffee)` | 2,879 segs |
| Space before paren | `ஹாய் (hi)` | 1,916 segs |
| **Square brackets** | `டவுட் [doubt]` | 521 glosses / 76 segs / 8 videos |
| **Curly braces** | `ट्राई {try}` | 10 glosses / 3 segs / 1 video |
| **Doubled / unbalanced parens** | `((Stardom))`, `ଫାସିଲିଟିଜ((facilities)` | 29 segs |
| **Native suffix re-attached after gloss** | `चॅनल(channel)-वर`, `ସେକ୍ସନରେ(section)-ରେ` | 1,831 |
| Numeral gloss | `एक(1)`, `दीड(1.5)` | 1,300 |
| Acronym gloss | `(CCBK)`, `(MTB)` | 331 |

A naive `re.sub(r'\([^)]*\)', '', text)` corrupts the 29 unbalanced segments,
leaves a dangling `-suffix` on 1,831 more, and misses the square- and
curly-bracket variants entirely — and **16 segments across 8 videos mix more
than one bracket style**, so the styles cannot be handled per-video either. Punctuation includes the Devanagari
danda `।` (1,706) alongside `.` (1,891), `,` (743) and `?` (531), so
normalization has to strip both ASCII and Indic punctuation.

---

# Step 1 — Audio extraction

Download each YouTube window, trim to `[start_sec, end_sec]`, publish as
**16 kHz mono WAV of exactly `round(duration × 16000)` samples**, checkpointed
to Google Drive with skip-if-exists.

**Architecture.** This notebook is orchestration only — config, stage toggles,
calls and display. All logic lives in the `sarvam_diar/` package beside it, so
it is reviewable outside the notebook and reusable across Steps 2–4.
`%autoreload` picks up edits to those modules without restarting the runtime.

**Two invariants the code is strict about:**

1. **Boundaries are exact.** Every published WAV holds exactly
   `round((end_sec - start_sec) * 16000)` samples, so each clip's timeline is
   exactly `[0, duration]` and the ground-truth timestamps — which are relative
   to `start_sec` — index straight into it with no offset arithmetic anywhere
   downstream. `DUR_TOL_SEC` is only a *diagnostic* that decides whether a fetch
   is trustworthy; it never defines the published boundary.
2. **A file existing proves nothing.** A Colab runtime can die mid-copy, so a
   clip counts as done only when its sidecar exists (written *last*, as the
   commit marker) **and** the WAV probes as 16 kHz mono `pcm_s16le` with the
   exact expected sample count.

## 1.0 — Setup

Installs `yt-dlp` (keep it current: YouTube extractor breakage is the single
most common cause of failure), mounts Drive, and puts `sarvam_diar/` on the
path. Everything here is cheap and safe to re-run.

In [ ]:
# --- dependencies -----------------------------------------------------------
# Everything is installed HERE, before any work, because pyannote pulls a newer
# numpy than Colab ships. Upgrading numpy under a live kernel leaves the process
# with half-old, half-new modules and the next numpy-touching import dies with
# `cannot import name '_center' from 'numpy._core.umath'`. Installing first
# means the one restart happens before anything expensive has run.
%pip install -q --upgrade yt-dlp
%pip install -q gdown "pyannote.audio>=4.0" "pyannote.metrics" itables

# Did an install replace a module this interpreter already imported? A fresh
# subprocess reads what is on disk; the running process reports what it loaded.
# If they disagree, this session is stale no matter what the imports look like.
import subprocess, sys
_stale = []
for _mod in ("numpy", "scipy"):
    try:
        _loaded = __import__(_mod).__version__
    except Exception:
        continue
    _disk = subprocess.run([sys.executable, "-c", f"import {_mod};print({_mod}.__version__)"],
                           capture_output=True, text=True).stdout.strip()
    if _disk and _disk != _loaded:
        _stale.append(f"{_mod}: loaded {_loaded}, on disk {_disk}")

if _stale:
    print("=" * 70)
    print("RESTART THE RUNTIME, then run this cell again.")
    print("  Runtime > Restart session   (Ctrl+M .)")
    print("\nAn install replaced a package this session had already imported:")
    for _s in _stale:
        print("   ", _s)
    print("\nNothing is lost -- every stage is checkpointed to Drive, so the")
    print("cells below re-run in seconds after the restart.")
    print("=" * 70)
    raise SystemExit("restart required")

import os, shutil, subprocess, sys
from pathlib import Path

for tool in ("ffmpeg", "ffprobe", "yt-dlp", "git"):
    path = shutil.which(tool)
    print(f"{tool:9s} {path or 'MISSING'}")
    assert path, f"{tool} not found on PATH"

# --- Drive ------------------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")

# --- code ------------------------------------------------------------------
# This notebook is orchestration only; the logic lives in the sarvam_diar
# package. On Colab it is cloned fresh from GitHub into local disk (not Drive --
# git over the Drive FUSE mount is slow and flaky). The repo is small, so a
# clone costs a second or two and guarantees the code matches the notebook.
REPO_URL = "https://github.com/ParvGoyal08/MultilingualASR.git"
CODE_DIR = Path("/content/sarvam-assignment")

def sync_code():
    """Clone or fast-forward the package. Returns the checkout path."""
    if (CODE_DIR / ".git").exists():
        r = subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only", "-q"],
                           capture_output=True, text=True)
        if r.returncode:
            # Local checkout diverged or is broken -- a fresh clone is always
            # correct here because nothing is ever edited on the Colab side.
            print("pull failed, re-cloning:", r.stderr.strip()[:200])
            shutil.rmtree(CODE_DIR, ignore_errors=True)
        else:
            return CODE_DIR
    subprocess.run(["git", "clone", "-q", REPO_URL, str(CODE_DIR)], check=True)
    return CODE_DIR

if IN_COLAB:
    PROJECT_DIR = sync_code()
    sha = subprocess.run(["git", "-C", str(PROJECT_DIR), "log", "-1", "--format=%h  %s"],
                         capture_output=True, text=True).stdout.strip()
    print(f"\ncode @ {sha}")
else:
    # Running locally: use the folder this notebook sits in.
    PROJECT_DIR = next((p for p in (Path.cwd(), Path.cwd().parent)
                        if (p / "sarvam_diar" / "__init__.py").exists()), None)
    assert PROJECT_DIR, "sarvam_diar/ not found next to the notebook"

sys.path.insert(0, str(PROJECT_DIR))
print("PROJECT_DIR", PROJECT_DIR)

# Drop any previously-imported sarvam_diar modules so the freshly pulled code is
# what actually gets imported. This is the mechanism that makes re-running this
# cell pick up a new commit -- without it Python serves the old modules from
# sys.modules and a `git pull` appears to do nothing.
for _m in [k for k in list(sys.modules)
           if k == "sarvam_diar" or k.startswith("sarvam_diar.")]:
    del sys.modules[_m]

# autoreload is a convenience on top of that (it also catches edits made
# in-place). Colab's IPython ships a version whose autoreload does
# `from imp import reload`, and `imp` was removed in Python 3.12 -- so this is
# best-effort and the purge above is what we actually rely on.
try:
    # Imported rather than relied on as an injected builtin, so editors can
    # resolve it; at runtime this is the same object IPython injects.
    from IPython import get_ipython

    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    print("autoreload: on")
except Exception as exc:
    print(f"autoreload unavailable ({type(exc).__name__}) -- re-run this cell "
          "after a push to pick up new code")

import pandas as pd

import sarvam_diar
from sarvam_diar import analysis, config, data, extraction, reference, utils
print("sarvam_diar", sarvam_diar.__version__)

## 1.1 — Config and stage flags

One visible cell holding every path and toggle. Each stage checkpoints to Drive,
so the normal workflow is: enable one stage, let it finish, flip it off, move
on. Re-running the notebook top-to-bottom then costs nothing but a few file
reads — which is what makes it survivable on a runtime that disconnects every
90 minutes.

In [ ]:
from sarvam_diar.config import Config, StageFlags

# Durable storage. On Colab this defaults to Drive so it survives a runtime
# reset; work_dir is fast local scratch (ffmpeg never writes into the Drive
# FUSE mount -- it is slow and gives no write atomicity).
cfg = Config.create(
    root=None,             # None -> Drive on Colab, ./pipeline_out locally
    # force_client="android",   # pin a yt-dlp player client to skip the ladder search
    # cookies_file=None,        # or drop a cookies.txt in <root>/data/ for bot-gated videos
)

flags = StageFlags(
    run_extraction=True,       # Step 1
    build_reference=True,      # scoring reference (cheap, safe to leave on)
    run_diarization=True,      # Step 2 -- gates BOTH the smoke run (2.1)
                               #           and the full sweep (2.2)
    run_asr=False,             # Step 3 (still a stub)
    run_refinement=False,      # Step 4 (still a stub)

    force_redo=False,          # re-run enabled stages from scratch, ignoring
                               # checkpoints. Honoured by extraction, reference
                               # AND diarization. No need to delete anything.
    retry_failed_only=False,   # only re-attempt non-permanent failures
    limit=None,                # e.g. 3 for a smoke run
    only_clip_ids=None,        # e.g. ["T3I2T-cfhG4"] to target specific clips
)

print(cfg.describe())
print(flags.describe())

### 1.1b — One-time: put your credentials in Drive

Secrets live in a `.env` file, not in the notebook — this notebook is committed
to a public repo, so anything typed into a cell or left in a cell's output gets
published.

`.env` is gitignored, which also means it does **not** arrive with the git
clone. On Colab it has to live in Drive. Run the cell below once; it prompts for
each value with `getpass`, so nothing is echoed to the screen or saved into the
`.ipynb`. It merges into any existing `.env` rather than overwriting it.

Skip this cell if `HF token: found` already printed in the Step 2 setup cell.

In [ ]:
# Run once. Writes MyDrive/sarvam_diarization/.env, which persists across
# runtimes -- you should not need this again on this Drive.
from getpass import getpass

print("target:", cfg.dotenv_path)
existing = utils.parse_dotenv(cfg.dotenv_path.read_text()) if cfg.dotenv_path.exists() else {}
print("already set:", sorted(existing) or "nothing")

# HF_TOKEN  -> gated pyannote models (Step 2). hf.co/settings/tokens, read scope.
# SARVAM_API_KEY -> Sarvam ASR (Step 3). Leave blank to skip either one.
for key, why in (("HF_TOKEN", "gated pyannote models, Step 2"),
                 ("SARVAM_API_KEY", "Sarvam ASR, Step 3")):
    value = getpass(f"{key} ({why}) -- blank to skip: ").strip()
    if value:
        utils.set_dotenv_value(cfg.dotenv_path, key, value)
        print(f"  {key} written ({len(value)} chars)")
    else:
        print(f"  {key} skipped")

now = utils.parse_dotenv(cfg.dotenv_path.read_text()) if cfg.dotenv_path.exists() else {}
print("\n.env now holds:", sorted(now))

## 1.2 — Load the CSV and parse the ground truth

The CSV is fetched to Drive once and reused (skip-if-exists). Parsing asserts
that the two label columns agree segment-for-segment, and drops the two corrupt
segments documented in Step 0 with an explicit record of what was excluded.

In [ ]:
df = data.load_segments_csv(cfg)
clips = data.parse_ground_truth(df)

print(f"{len(df)} rows, {df.video_id.nunique()} unique video_ids, "
      f"{sum(len(c.segments) for c in clips)} segments")

# The duplicated video_id is why clip_id is composite, not just video_id.
dupes = df[df.video_id.duplicated(keep=False)]
print("\nSame video, two windows -> two distinct clip_ids:")
display(dupes[["video_id", "start_sec", "end_sec", "clip_id"]])

gt = data.clips_to_frame(clips)
display(gt.head(3))

## 1.3 — Dataset profile

Recomputes every number quoted in Step 0 from the file itself and writes
`results/dataset_profile.json`. Run it after any change to the parser: if a
number here moves, the Step 0 notes are stale.

In [ ]:
import json
profile = data.profile_dataset(clips, cfg)

print("clips              ", profile["n_rows"], "rows /", profile["n_unique_video_ids"], "videos")
print("segments           ", profile["n_segments"], f"({profile['n_dropped_segments']} dropped as malformed)")
print("audio              ", profile["clip_duration_sec"]["total_hours"], "h")
print("speakers per clip  ", profile["speakers"]["per_clip_distribution"])
print("overlap            ", f"{profile['overlap']['overlap_frac_of_corpus']:.2%} of corpus time,",
      profile["overlap"]["n_clips_without_overlap"], "clips with none")
print("GT beyond end_sec  ", profile["gt_boundary"]["n_clips_gt_beyond_end_sec"], "clips,",
      f"median +{profile['gt_boundary']['overrun_median_sec']}s,",
      f"max +{profile['gt_boundary']['overrun_max_sec']}s")
print("scripts            ", profile["language"]["script_distribution"])
print("non-speech tags    ", profile["transcript"]["nonspeech_tags"])
print("code-switch glosses", profile["transcript"]["n_code_switch_glosses"],
      "in", profile["transcript"]["n_segments_with_gloss"], "segments")

print("\nDropped ground-truth segments:")
print(json.dumps(profile["dropped_segments"], indent=2, ensure_ascii=False))

### 1.3b — What is actually present under `cfg.root`?

Step 1 may have run on a different machine (it did here: extraction locally,
because Colab's IPs are bot-gated by YouTube). The artifacts then arrive by a
manual copy, which can be partial — so check before depending on them, rather
than discovering it as a traceback three cells later.

In [ ]:
inv = extraction.inventory(cfg, expected_clips=len(clips))
display(inv[["artifact", "present", "count", "expected"]])

missing = inv[~inv.present]
if len(missing):
    print("MISSING:")
    for a in missing.artifact:
        print("   ", a)
    print(f"\nCopy the CONTENTS of your local `local_out/` folder into {cfg.root}\n"
          "so it holds audio_16k/, meta/, results/, reference/ and data/.")
else:
    print("all Step 1 artifacts present")

# Exactly which clips lack usable audio, largest first. A manual copy that
# stopped part-way shows up here as the biggest files -- re-upload just these
# rather than the whole 1.3 GB.
gaps = extraction.missing_audio(cfg, clips)
if len(gaps):
    print(f"\n{len(gaps)} clip(s) without usable audio, "
          f"{gaps.expect_bytes.sum()/1e6:.0f} MB to transfer:")
    display(gaps[["clip_id", "state", "expect_mb", "duration_sec"]])
    print("GUVrL5ltiP4 is expected here -- that source video is deleted.")
    print("For anything else: copy those WAVs into", cfg.audio_dir)
    print("and keep run_extraction=False -- Colab cannot re-download them.")
else:
    print("\nevery clip has a usable WAV")

# 99 not 100 is expected: GUVrL5ltiP4 is a deleted video.
short = inv[(inv["count"] < inv.expected) & inv.present]
if len(short):
    print("\nfewer than expected (99/100 is normal -- one source video is deleted):")
    display(short[["artifact", "count", "expected"]])

## 1.4 — Extract

Per clip: resolve the media URL, range-fetch only `[start_sec, end_sec]` with
ffmpeg, force the exact sample count, publish atomically to Drive, then write
the sidecar as the commit marker.

**Fetch strategy.** Which YouTube player client yields a *downloadable* format
is not stable — it varies by network, video and yt-dlp version, and the client
with the best formats is often not a working one. Measured while building this:
yt-dlp's default rotation offers opus audio-only (itag 251) but those URLs 403
without a PO token, while `android` offers only itag 18 (muxed 360p) and
downloads fine. So the ladder is *walked*, not guessed, and whichever client
works is reused for the remaining clips — only the first clip pays for the
search. Tier A range-fetches just the window; tier B downloads the full track
and trims locally, and is used when tier A fails or its output drifts beyond
`DUR_TOL_SEC`. Both were verified to produce **byte-identical** audio.

Safe to interrupt and re-run — finished clips are skipped. If Colab hits
YouTube's bot gate, drop a Netscape-format `cookies.txt` into `<root>/data/`.

In [ ]:
if flags.run_extraction:
    results = extraction.run(cfg, clips, flags)
else:
    # The per-clip sidecars in meta/ are the commit markers and the authority;
    # step1_extraction.csv is derived. If they disagree -- e.g. a stale CSV from
    # an earlier failed run survived while the audio was copied in from another
    # machine -- trust the sidecars and repair the CSV.
    results, recon = extraction.reconcile(cfg, clips)
    print(f"extraction skipped. {recon['action']}: "
          f"csv={recon['csv_ok_rows']} ok, sidecars={recon['sidecar_ok_rows']} ok")
    if recon.get("note"):
        print("\n" + recon["note"])
    if int((results.status == "ok").sum()) == 0:
        raise FileNotFoundError(
            f"No successfully extracted clips under {cfg.root}.\n"
            "Neither the results CSV nor the per-clip sidecars in meta/ show any.\n"
            "Copy the CONTENTS of your local `local_out/` folder into that "
            "directory (see the inventory cell above)."
        )

## 1.5 — Report

Per-video results, the run summary, and the failure breakdown. `all_exact` is
the one that matters: it asserts every published WAV holds exactly the requested
number of samples.

In [ ]:
import json

# The CSV is the source of truth; the summary JSON is optional enrichment. If
# Step 1 ran elsewhere and only part of results/ was copied across, report from
# what is here instead of failing.
summary = utils.read_json(cfg.extraction_summary)
if summary:
    print(json.dumps({k: summary[k] for k in
                      ("totals", "audio_contract", "download_tiers", "player_clients",
                       "formats", "error_classes", "session_counts") if k in summary},
                     indent=2, ensure_ascii=False))
    assert summary["audio_contract"]["all_exact"], "some WAVs are not exactly the requested length"
else:
    print(f"note: {cfg.extraction_summary.name} not found -- reporting from "
          f"{cfg.extraction_csv.name} instead.\n")

done = results[results.status == "ok"]
hours = pd.to_numeric(done.requested_dur_sec, errors="coerce").sum() / 3600
gb = pd.to_numeric(done.file_size_bytes, errors="coerce").sum() / 1e9
print(f"{len(done)} extracted / {len(results)} rows | {hours:.2f} h | {gb:.2f} GB")

# Re-derive the exact-length contract from the CSV, so it is checked even when
# the summary JSON is absent.
n = pd.to_numeric(done.n_samples, errors="coerce")
exp = pd.to_numeric(done.n_expected_samples, errors="coerce")
print(f"exact-length contract: {int((n == exp).sum())}/{len(done)} rows exact")
assert (n == exp).all(), "some WAVs are not exactly the requested sample count"

display(done[["clip_id", "requested_dur_sec", "n_samples", "n_expected_samples",
              "raw_delta_sec", "pad_samples", "trim_samples", "download_tier",
              "player_client", "ytdlp_format_id", "attempts", "elapsed_sec",
              "n_gt_speakers", "gt_overlap_frac", "ref_lang_hint"]].head(20))

failed = results[results.status == "failed"]
if len(failed):
    print(f"\n{len(failed)} failed ({int(failed.permanent.sum())} permanent):")
    display(failed[["clip_id", "error_class", "permanent", "attempts", "error_msg"]])
else:
    print("\nno failures")

attempts_log = pd.DataFrame(utils.read_jsonl(cfg.failures_jsonl))
if len(attempts_log):
    print(f"\nfailed attempts by class and client ({len(attempts_log)} total):")
    display(attempts_log.groupby(["error_class", "player_client"]).size()
            .rename("n").reset_index())

## 1.6 — Verification

Three checks, in order of how much they would cost to get wrong:

1. **Audit** — re-probes every published WAV against the exact-length contract
   and reports orphans, so a corrupted checkpoint is caught before Step 2 reads
   it as truth.
2. **Alignment** — the important one. Ground-truth timestamps are relative to
   `start_sec`, so a one-second trim offset would silently wreck DER for that
   clip with no visible symptom. Slices the first few ground-truth segments out
   of a published WAV and renders them next to their reference text: if the
   offset is wrong, the audio will not match the words.
3. **Idempotency** — re-running the extract cell must skip everything and touch
   the network zero times.

In [ ]:
# --- 1. audit every published WAV -------------------------------------------
audit = extraction.audit(cfg, clips)
attempted = audit[audit.clip_id.isin(results[results.status == "ok"].clip_id)]
print(f"audit: {attempted.valid.sum()}/{len(attempted)} extracted clips pass the exact-length contract")
bad = attempted[~attempted.valid]
if len(bad):
    display(bad)

orphans = {p.stem for p in cfg.audio_dir.glob("*.wav")} - {c.clip_id for c in clips}
print("orphan WAVs on Drive:", sorted(orphans) or "none")

In [ ]:
# --- 2. alignment spot-check ------------------------------------------------
# If the trim offset were wrong, the words below would not match the audio.
from IPython.display import Audio, display as ipy_display

ok_ids = results[results.status == "ok"].clip_id.tolist()
if not ok_ids:
    print("no successfully extracted clips -- see the inventory cell (1.3b)")
    print("status counts:", results.status.value_counts().to_dict())
else:
    clip = next(c for c in clips if c.clip_id == ok_ids[0])
    wav = cfg.wav_path(clip.clip_id)     # resolved against THIS machine's cfg
    if not wav.exists():
        print(f"results say {clip.clip_id} is extracted, but {wav} is missing.\n"
              "The audio_16k/ folder did not come across -- see cell 1.3b.")
    else:
        print(f"{clip.clip_id}  |  {clip.stats['lang_script']}  |  "
              f"{clip.stats['n_gt_speakers']} speakers  |  {len(clip.segments)} segments")
        for seg in clip.segments[:4]:
            print(f"\n[{seg.start:7.2f} - {seg.end:7.2f}]  {seg.speaker}")
            print(f"   {seg.text[:160]}")
            ipy_display(Audio(extraction.read_wav_window(wav, seg.start, seg.end),
                              rate=cfg.sample_rate))

In [ ]:
# --- 3. idempotency ---------------------------------------------------------
# Must report every already-extracted clip as skipped, with no network traffic.
_ = extraction.run(cfg, clips, StageFlags(only_clip_ids=ok_ids[:3]))

---

# Step 1.7 — Build the scoring reference

Everything so far has been pipeline-side. This section builds the **scoring
reference** from the ground truth, and puts a hard wall between the two.

The brief is unambiguous:

> **The ground truth is never an input to your pipeline.** `diarization_segments`
> and `asr_segments` are only for computing the final scores.

### Why the reference has to be built rather than scored raw

| Problem | Scale | Consequence if ignored |
|---|---|---|
| GT runs past the audio window | 85/100 clips, median +1.43 s | hypothesis silence scored against speech that is not in the file |
| Same-speaker intervals overlap themselves | 140 pairs | one speaker's second counted twice in the DER denominator |
| Corrupt segments | 2 (`end < start`, zero-length) | undefined turn geometry |
| **Code-switch gloss is not speech** | **24,914 of 149,121 tokens — 16.7%** | **a ~17% cpWER floor that says nothing about ASR quality** |

### What is deliberately *not* done

No overlap removal, no forgiveness collar as the headline metric, no
minimum-duration filter, no per-system normalization. Overlap in particular is
scored in full — the brief requires it.

### The line between normalization and cheating

The normalizer is legitimate because it is (1) a pure function of one text
string with no access to the other side, (2) applied **identically** to
reference and hypothesis, (3) fixed and versioned before any system output was
seen, and (4) documented with its corpus-wide token impact. `strip_gloss` is the
only asymmetry and it is inert — hypotheses contain no glosses, which the
verification cell asserts.

In [ ]:
# Ground truth in -> ClipInput (pipeline-safe) + ClipReference (scoring only).
inputs, ref_clips = data.split_reference(clips, results, cfg=cfg)
print(f"{len(inputs)} pipeline inputs (extracted clips only), {len(ref_clips)} reference clips")
print("ClipInput fields:", sorted(data.ClipInput.__dataclass_fields__))

if flags.build_reference:
    manifest = reference.run(cfg, clips, force=flags.force_redo)
else:
    manifest = pd.read_csv(cfg.reference_manifest)

display(manifest[["clip_id", "n_turns", "n_speakers", "speaker_time_sec", "overlap_sec",
                  "overlap_frac", "n_utterances", "n_ref_tokens",
                  "n_segments_dropped_by_uem", "n_segments_truncated_by_uem",
                  "n_same_speaker_merges"]].head(10))

print(f"\nturns {int(manifest.n_turns.sum())} | utterances {int(manifest.n_utterances.sum())} "
      f"| reference tokens {int(manifest.n_ref_tokens.sum())}")
print(f"UEM crop: {int(manifest.n_segments_dropped_by_uem.sum())} segments dropped, "
      f"{int(manifest.n_segments_truncated_by_uem.sum())} truncated, "
      f"{int(manifest.n_same_speaker_merges.sum())} same-speaker merges")
print("reference speaker counts:", manifest.n_speakers.value_counts().sort_index().to_dict())

In [ ]:
# --- normalization report + the golden table ---------------------------------
import json
report = reference.normalization_report(clips, cfg)
print(json.dumps(report, indent=2, ensure_ascii=False))

print("\nGolden cases -- every hard variant found in the corpus, eyeball these:\n")
for case in reference.GOLDEN_CASES:
    print("RAW :", case)
    print("NORM:", reference.normalize_text(case) or "(empty -- non-speech, excluded from WER)")
    print()

## 1.8 — Reference and leak-guard verification

Fifteen assertions. The ones that matter most:

* **Cropping must not change the speaker set** — otherwise speaker-count
  accuracy is measured against a reference the audio cannot support.
* **Cross-speaker overlap must survive at 7.13%** — a regression here means the
  reference quietly stopped scoring the overlap the brief requires.
* **The normalizer must be idempotent** and must never empty a speech segment.
* **`ClipInput` must carry no ground-truth field**, and the DataFrame guard must
  reject `n_gt_speakers` / `ref_lang_hint`. That column pair is exactly what
  would otherwise become pyannote's `num_speakers` or Whisper's `language`.

In [ ]:
import re, collections
refs = {c.clip_id: reference.build_reference(c) for c in clips}
speech = [s for c in clips for s in c.segments if s.is_speech]
failures = []

def check(name, ok, detail=""):
    print(f"  {'PASS' if ok else 'FAIL'}  {name}{'  ' + detail if detail else ''}")
    if not ok:
        failures.append(name)

print("=== reference integrity ===")
check("no turn outside the UEM",
      not [t for r in refs.values() for t in r.turns if t.end > r.uem[1] + 1e-6 or t.start < -1e-9])
# Compare through rttm_safe: reference turn labels are normalised to the
# RTTM-legal form (`Speaker A` -> `Speaker_A`) at build time, while raw segments
# keep the CSV spelling. The property under test is that cropping loses no
# speaker, not how the label is spelled.
check("cropping preserves the speaker set",
      all({reference.rttm_safe(s.speaker) for s in c.segments}
          == {t.speaker for t in refs[c.clip_id].turns} for c in clips),
      f"{len(clips)}/{len(clips)}")
disjoint = True
for r in refs.values():
    per = collections.defaultdict(list)
    for t in r.turns:
        per[t.speaker].append((t.start, t.end))
    for v in per.values():
        v.sort()
        disjoint &= all(v[i][0] >= v[i - 1][1] - 1e-9 for i in range(1, len(v)))
check("same-speaker intervals disjoint after union", disjoint)
frac = sum(r.stats["overlap_sec"] for r in refs.values()) / sum(c.duration for c in clips)
check("cross-speaker overlap preserved", abs(frac - 0.0713) < 0.002, f"{frac:.4%} of corpus")
check("RTTM round-trips exactly",
      all(reference.parse_rttm(reference.to_rttm(r)) ==
          [data.Turn(t.speaker, round(t.start, 6), round(t.end, 6)) for t in r.turns]
          for r in refs.values()))

print("\n=== normalizer ===")
norm = [reference.normalize_text(s.text) for s in speech]
check("idempotent", all(reference.normalize_text(n) == n for n in norm), f"{len(speech)} segments")
check("no speech segment normalizes to empty", all(norm))
check("token count matches the report",
      sum(len(n.split()) for n in norm) == report["tokens"]["gloss_stripped_final"],
      f"{sum(len(n.split()) for n in norm)}")
check("reference is essentially pure native script",
      len(re.findall(r'[a-z]+', ' '.join(norm))) <= 40,
      f"{len(re.findall(r'[a-z]+', ' '.join(norm)))} residual Latin tokens")
gloss_free = ["नमस्कार मी गौरव जोशी आहे", "இது ஒரு சோதனை", "hello this is a test"]
check("gloss step is inert on gloss-free text (ref/hyp symmetry)",
      all(reference.normalize_text(h, True) == reference.normalize_text(h, False) for h in gloss_free))

print("\n=== leak guard ===")
fields = set(data.ClipInput.__dataclass_fields__)
check("ClipInput has no segments attribute", "segments" not in fields)
check("ClipInput carries no gt_/ref_ field", not utils.reference_fields(fields))
for bad in ("n_gt_speakers", "ref_lang_hint"):
    try:
        utils.assert_no_reference_fields(pd.DataFrame(columns=["clip_id", bad]))
        check(f"guard rejects {bad}", False)
    except AssertionError:
        check(f"guard rejects {bad}", True)
utils.assert_no_reference_fields(pd.DataFrame(columns=["clip_id", "wav_path"]))
check("guard passes clean columns", True)
check("split_reference yields ClipInput only",
      all(isinstance(v, data.ClipInput) for v in inputs.values()), f"{len(inputs)} inputs")

assert not failures, f"reference verification failed: {failures}"
print("\nALL CHECKS PASSED")

---

# Step 2 — Baseline diarization benchmarking

Two pyannote pipelines over the extracted clips: **community-1** (primary) and
**speaker-diarization-3.1** (the baseline it replaced). Same library, one code
path, and the delta between them is the version-over-version comparison.

### Scoring rules, and why

The brief is explicit: *"Do NOT ignore overlapping speech regions when computing
metrics. Overlap must be scored."* So the headline numbers use **collar 0.0 and
`skip_overlap=False`** — score everything, forgive nothing. That matters here
more than usual: **7.13% of scored time has ≥2 speakers active** and only 9 of
100 clips have none. A collared, overlap-forgiving variant is reported
alongside for comparability with published figures, never instead.

### No speaker-count hints

`num_speakers` / `min_speakers` / `max_speakers` are never passed. The pipeline
has to estimate the count itself — that estimate is exactly what speaker-count
accuracy measures, and feeding it the reference count is the classic leak.
`data.ClipInput` makes this structural: the type handed to `diarization.run()`
has no speaker field to leak.

### Credentials

Both pipelines are gated. The token is read from **`.env`**, never from a
notebook cell. `.env` is gitignored so it never reaches the public repo — which
also means it does not arrive with the git clone, so on Colab it has to live in
Drive at `MyDrive/sarvam_diarization/.env`:

```
HF_TOKEN="hf_..."
SARVAM_API_KEY="..."      # Step 3
```

### What to expect

community-1 reports ~11–20% DER on standard benchmarks. **Ours will be worse.**
This is 9-language Indic code-switched YouTube audio, 20% of reference segments
are under 0.5 s, and we score with no collar and full overlap. **25–40% DER
would be unsurprising and is not a bug.** A DER near 0, or above ~70%, probably
is.

In [ ]:
# pyannote was installed in cell 1.0, deliberately: installing it here would
# upgrade numpy under a live kernel and break the session. Nothing is installed
# in this cell -- it only verifies.
#
# pip's resolver warnings during 1.0 (numba wanting numpy<2.1, google-adk wanting
# older opentelemetry) are about Colab preinstalls, not our dependencies:
# pyannote.audio declares no numba/librosa dependency and importing the pipeline
# pulls in neither. This check is what actually decides usability.
import sys, importlib

for mod in ("torch", "pyannote.audio", "pyannote.metrics", "pyannote.core"):
    m = importlib.import_module(mod)
    print(f"  {mod:18s} {getattr(m, '__version__', '?')}")

from pyannote.audio import Pipeline                      # noqa: F401
from pyannote.audio.pipelines import SpeakerDiarization  # noqa: F401
from pyannote.metrics.diarization import DiarizationErrorRate  # noqa: F401
leaked = sorted(m for m in sys.modules if m.split(".")[0] in ("numba", "llvmlite"))
print(f"  numba pulled in by pyannote: {leaked or 'no'}")
print("  imports OK -- resolver warnings above do not affect this pipeline\n")

from sarvam_diar import diarization, evaluation

# --- GPU ---------------------------------------------------------------------
dev = diarization.device_report()
print("device:", dev)
if not dev["cuda"]:
    print("\nWARNING: no GPU. Runtime > Change runtime type > T4 GPU, then re-run.")

# --- HF token from .env -------------------------------------------------------
# community-1 and 3.1 are gated. The token is read from a .env file, never from
# a cell -- this notebook is published to a public repo.
#
# .env is gitignored, so it does NOT arrive with the git clone. On Colab put a
# copy in Drive at the path printed below. Same file also carries SARVAM_API_KEY
# for Step 3.
tok = diarization.resolve_token(cfg)
print("expected .env:", cfg.dotenv_path)
# Only ever print a masked form: this output is saved into the .ipynb.
print("HF token     :", f"found ({tok[:5]}...{tok[-3:]}, {len(tok)} chars)" if tok
      else f"MISSING -- run cell 1.1b to write it to {cfg.dotenv_path}")
if tok:
    print("               you must also accept conditions at:")
    for k, repo in config.DIARIZATION_MODELS.items():
        print(f"                 https://hf.co/{repo}")

# --- metric semantics probe ---------------------------------------------------
# Re-derives pyannote.metrics' behaviour at runtime rather than trusting the
# docs, which pin down none of it. If a library upgrade renames a component or
# switches accumulation from pooling to averaging, this catches it here instead
# of silently shifting every reported number.
probe = evaluation.probe_metric_semantics()
for k, v in probe.items():
    print(f"  {k}: {v}")
assert not probe["problems"], probe["problems"]

## 2.1 — Smoke run: measure throughput before committing to the full sweep

Three clips chosen to span the duration range — RTF is not constant, so a smoke
set of three short clips would give a misleadingly optimistic projection. Fixed
per-call overhead dominates on short clips; memory pressure shows up on long
ones, and the longest clip in the corpus is 30 minutes.

The projection this prints is the gate on whether to run the full sweep.

In [ ]:
# ClipInput only -- no speaker counts, no language, no ground truth.
inputs, ref_clips = data.split_reference(clips, results, cfg=cfg)
by_id = {c.clip_id: c for c in inputs.values()}
print(f"{len(inputs)} extracted clips available to the pipeline")

# short / mid / long, so the RTF spread is visible
ordered = sorted(inputs.values(), key=lambda c: c.duration)
smoke = [ordered[0], ordered[len(ordered) // 2], ordered[-1]]
for c in smoke:
    print(f"  smoke: {c.clip_id:<28} {c.duration:7.0f}s")

if not flags.run_diarization:
    raise SystemExit("flags.run_diarization is False -- set it True in cell 1.1")

smoke_df = diarization.run(cfg, smoke,
                           StageFlags(force_redo=flags.force_redo))
display(smoke_df[["model", "clip_id", "status", "n_turns", "n_speakers_hyp",
                  "elapsed_sec", "rtf"]])

tput = diarization.throughput_report(smoke_df)
print("\nMEASURED throughput (not an estimate):")
display(tput)
print(f"GPU: {dev['device_name']}")
print("\n^ projected_full_sweep is per model, for all 12.26 h. Decide from this "
      "whether to run the full sweep now, trim the roster, or change runtime tier.")

## 2.2 — Full sweep

Checkpointed per clip: RTTM written first, sidecar JSON last as the commit
marker. Safe to interrupt — finished clips are skipped and a Colab disconnect
costs only the clip in flight.

Set `LIMIT = None` to run everything.

In [ ]:
LIMIT = None          # e.g. 10 to extend gradually; None runs all clips

if not flags.run_diarization:
    print("=" * 68)
    print("FULL SWEEP SKIPPED -- flags.run_diarization is False.")
    print("Only the 3 smoke clips from 2.1 have been diarized, and LIMIT is")
    print("not even consulted. Set run_diarization=True in cell 1.1, re-run it,")
    print("then re-run this cell.")
    print("=" * 68)
else:
    n_clips = len(inputs) if LIMIT is None else min(LIMIT, len(inputs))
    print(f"sweeping {n_clips} clips x {len(config.DIARIZATION_MODELS)} models "
          f"= {n_clips * len(config.DIARIZATION_MODELS)} runs "
          f"(already-finished clips are skipped)\n")

    hyp_df = diarization.run(cfg, list(inputs.values()),
                             StageFlags(limit=LIMIT, force_redo=flags.force_redo))
    ok = hyp_df[hyp_df.status == "ok"]
    print(f"\n{len(ok)} hypotheses / {len(hyp_df)} attempts")

    # Did the sweep actually cover everything it was asked to?
    expected = n_clips * len(config.DIARIZATION_MODELS)
    if len(ok) < expected:
        print(f"WARNING: expected {expected} successful runs, got {len(ok)}. "
              "Check the failures below and the LIMIT above.")
    if (hyp_df.status == "failed").any():
        display(hyp_df[hyp_df.status == "failed"][["model", "clip_id", "error_class"]])

## 2.2b — Adopt DiariZen (optional)

DiariZen cannot share this environment: it pins `numpy==1.26.4` while
`pyannote.audio` 4.x needs numpy 2.x. It therefore runs in
`notebooks/diarizen_runner.ipynb`, in its own session, and hands over plain
RTTM files.

Attach that runner's output as an Input, set `DIARIZEN_RTTM` below, and run. The
import writes the same RTTM + sidecar pair a native run writes, so every stage
after this — scoring, ranking, the error explorer — treats `diarizen-large`
identically to a model we ran in-process.

Skip this cell if you have not run the DiariZen notebook. Nothing downstream
requires it.


In [ ]:
# Point this at the DiariZen runner's RTTM folder, or leave None to skip.
# Attach the runner's output via Input > Add Input > Your Work.
DIARIZEN_RTTM = None      # e.g. "/kaggle/input/diarizen-rttm"

if DIARIZEN_RTTM is None:
    # Already-imported RTTMs stay on disk, so this cell is a no-op on a re-run
    # rather than something that has to be run every session.
    have = sum(1 for cid in inputs
               if diarization.is_done(cfg, "diarizen-large", cid))
    print(f"DIARIZEN_RTTM not set -- skipping import "
          f"({have} clips already imported on disk)")
else:
    src = Path(DIARIZEN_RTTM)
    if not src.is_dir():
        # Tolerate being pointed at the dataset root rather than the folder.
        nested = sorted(p.parent for p in Path(DIARIZEN_RTTM).rglob("*.rttm"))
        if nested:
            src = nested[0]
        else:
            raise FileNotFoundError(
                f"no .rttm files under {DIARIZEN_RTTM}. Attach the DiariZen "
                "runner's output and set DIARIZEN_RTTM to it.")
    imported = diarization.import_external_rttm(
        cfg, src, model="diarizen-large",
        clip_durations={c.clip_id: c.duration for c in inputs.values()},
    )
    print(f"imported {len(imported)} clips as 'diarizen-large' from {src}")
    if len(imported):
        display(imported[["clip_id", "n_turns", "n_speakers_hyp", "max_end"]].head())
    missing = [cid for cid in inputs
               if not diarization.is_done(cfg, "diarizen-large", cid)]
    if missing:
        print(f"WARNING: {len(missing)} clips have no DiariZen RTTM, e.g. {missing[:3]}")
        print("The sweep was incomplete -- resume it in the runner notebook.")


## 2.3 — Score: DER, JER, speaker-count accuracy

Corpus DER is computed by **pooling seconds**, not by averaging per-clip DER.
Averaging would weight a 50-second clip the same as a 30-minute one — a
different, and wrong, statistic. The probe in 2.0 confirmed pyannote pools too,
so both routes agree.

In [ ]:
# Load reference + hypotheses from the checkpoints, then score.
references = {cid: reference.load_reference(cfg, cid) for cid in inputs}
hypotheses = {}
for model in config.scored_models():
    for cid in inputs:
        if diarization.is_done(cfg, model, cid):
            hypotheses[(model, cid)] = diarization.load_hypothesis(cfg, model, cid)
print(f"scoring {len(hypotheses)} (model, clip) pairs")

metrics = evaluation.score_all(cfg, references, hypotheses)
corpus = evaluation.aggregate(metrics)

if not len(metrics):
    print("\nNothing to score yet -- no hypotheses on disk.")
    print("Run cells 2.1 / 2.2 first. If they failed with GatedRepoError, accept")
    print("the model conditions at the HuggingFace links printed in cell 2.0")
    print("(including pyannote/segmentation-3.0, which 3.1 pulls in).")
else:
    print("\n=== CORPUS (collar 0.0, overlap scored) ===")
    display(corpus[["model", "n_clips", "der", "der_fa_frac", "der_miss_frac",
                    "der_confusion_frac", "jer_mean", "speaker_count_accuracy",
                    "speaker_count_mae", "speaker_count_bias"]])

    print("=== lenient variant (collar 0.25, overlap skipped) -- secondary only ===")
    display(corpus[["model", "lenient_der", "lenient_jer_mean"]])

    print("=== per-clip (head) ===")
    display(metrics[["model", "clip_id", "der", "jer", "n_speakers_ref",
                     "n_speakers_hyp", "speaker_count_error"]].head(12))

## 2.4 — Where does the error actually come from?

DER alone does not say whether the system got the **number** of speakers wrong,
or got the number right and assigned the wrong one. Two analyses:

**(a) Stratified — leak-free.** Clips split by `n_hyp − n_ref`. Inside the
`exact` stratum no reference speaker is unmappable, so its confusion component
is *pure assignment error*. Over-estimation shows up as confusion plus false
alarm; under-estimation as missed detection plus confusion.

**(b) Oracle-count ablation — a diagnostic, quarantined.** Re-runs with
`num_speakers` set from the reference. The DER gap is what count estimation
costs, in DER points. This *does* feed ground truth to the model, so it is
off by default, writes to a separate `hypotheses_oracle/` tree, prefixes every
column `oracle_`, and must never be reported as system performance.

In [ ]:
if not len(metrics):
    print("no metrics yet -- see cell 2.3")
else:
    print("=== DER by speaker-count stratum (leak-free) ===")
    strata = evaluation.stratify_by_count_error(metrics)
    display(strata[["model", "stratum", "n_clips", "der", "der_fa_frac",
                    "der_miss_frac", "der_confusion_frac"]])
    print("In the `exact` stratum, der_confusion_frac is pure assignment error.\n")

    print("=== count error vs DER ===")
    display(evaluation.count_error_correlation(metrics))

    # pooling the strata must reproduce the headline number
    for model, sub in metrics.groupby("model"):
        pooled = evaluation.pool(sub.to_dict("records"))["der"]
        head = corpus.loc[corpus.model == model, "der"].iloc[0]
        assert abs(pooled - head) < 1e-9, f"{model}: strata do not reconcile"
    print("strata reconcile with the corpus DER\n")

    # --- oracle ablation, opt-in, DIAGNOSTIC ONLY ---------------------------
    if flags.run_oracle_count_ablation:
        oracle_counts = {cid: references[cid].n_speakers for cid in inputs}
        oracle_hyp_df = diarization.run(cfg, list(inputs.values()),
                                        StageFlags(limit=LIMIT, force_redo=flags.force_redo),
                                        oracle_counts=oracle_counts)
        oracle_hyps = {(m, cid): diarization.load_hypothesis(cfg, m, cid, oracle=True)
                       for m in config.scored_models() for cid in inputs
                       if diarization.is_done(cfg, m, cid, oracle=True)}
        oracle_metrics = evaluation.score_all(None, references, oracle_hyps)
        if len(oracle_metrics):
            print("=== ORACLE ABLATION -- diagnostic, NOT system performance ===")
            display(evaluation.oracle_gap(metrics, oracle_metrics))
    else:
        print("oracle ablation off (flags.run_oracle_count_ablation=False)")

    summary = evaluation.summarize(cfg, metrics)
    print(f"\nwrote {cfg.step2_metrics_csv}")
    print(f"wrote {cfg.step2_summary}")

## 2.5 — Error analysis: where the DER actually comes from

Eight rankings plus a head-to-head. Tables are interactive (sortable, searchable,
paginated) — click a column header to re-sort.

**Rate and contribution answer different questions.** A 50 s clip at DER 0.90 has
a terrible rate but is a rounding error in the corpus number; a 30-minute clip at
DER 0.30 can be a fifth of all the error in the benchmark. Every table below
carries both `der` (the rate) and `error_share` (this clip's share of the model's
total error seconds). **"Biggest contributors" is the list to act on** — those are
the clips where an improvement actually moves the headline.

Reading the error types:

| column | means | usually caused by |
|---|---|---|
| `der_miss_sec` | reference speech never detected | VAD/segmentation; overlapped and nested speech |
| `der_confusion_sec` | speech found, wrong speaker | clustering — too few/many clusters, or bad embeddings |
| `der_fa_sec` | speech emitted where reference has none | over-eager VAD — **or unannotated reference tails** |
| `overlap_der` | DER scored *only* where ≥2 speakers are active | the hardest condition; NaN where a clip has no overlap |

On false alarm specifically: `Cku_X_SL7qU` (90 s unannotated), `OxYCBQKZ3iY` (12 s)
and `CO_8ppdzq9U` (6 s) have tails the annotator never labelled, so a high FA
there may be the reference's fault rather than the model's.

In [ ]:
if not len(metrics):
    print("no metrics yet -- run 2.1/2.2 first")
else:
    enriched = analysis.enrich(metrics)

    # --- corpus-level: where does each model's error go? ---------------------
    analysis.show(analysis.error_composition(enriched),
                  "Error composition per model (% of that model's error seconds)")

    # --- the eight rankings --------------------------------------------------
    for caption, table in analysis.all_rankings(metrics, n=20).items():
        analysis.show(table, caption)

In [ ]:
if len(metrics):
    # --- head-to-head --------------------------------------------------------
    analysis.show(analysis.head_to_head_summary(enriched, "der"), "Head-to-head summary (DER)")
    analysis.show(analysis.model_comparison(enriched, "der"),
                  "Per-clip model comparison -- delta > 0 means the first model is worse")

    # --- concentration: do a few clips dominate? -----------------------------
    for model in sorted(enriched.model.unique()):
        p = analysis.pareto(enriched, model, n=15)
        analysis.show(p, f"Error concentration -- {model}")
        if len(p):
            print(f"   top {len(p)} clips = {p.cumulative_share.iloc[-1]:.1%} "
                  f"of {model}'s total error seconds")

### 2.6 — Error composition chart

Stacked bars: how each model's error seconds split between false alarm, missed
speech and confusion, for the clips contributing most. A tall bar that is mostly
purple is a clustering problem; mostly blue is a detection problem. They call for
different fixes in Step 4.

In [ ]:
if len(metrics):
    import matplotlib.pyplot as plt

    top = (enriched.sort_values("error_sec", ascending=False)
           .groupby("model").head(15))
    models = sorted(top.model.unique())
    fig, axes = plt.subplots(len(models), 1, figsize=(13, 5 * len(models)), squeeze=False)

    for ax, model in zip(axes[:, 0], models):
        sub = top[top.model == model].sort_values("error_sec")
        ax.barh(sub.clip_id, sub.der_fa_sec, label="false alarm", color="#4C9F70")
        ax.barh(sub.clip_id, sub.der_miss_sec, left=sub.der_fa_sec,
                label="missed", color="#3E7CB1")
        ax.barh(sub.clip_id, sub.der_confusion_sec,
                left=sub.der_fa_sec + sub.der_miss_sec, label="confusion", color="#8B5FBF")
        ax.set_title(f"{model} -- 15 clips contributing the most error")
        ax.set_xlabel("error seconds")
        ax.tick_params(axis="y", labelsize=8)
        ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    # overlap vs single-speaker: the condition that actually separates models
    fig, ax = plt.subplots(figsize=(7, 5))
    for model in models:
        sub = enriched[(enriched.model == model) & enriched.overlap_der.notna()]
        ax.scatter(sub.single_speaker_der, sub.overlap_der, alpha=0.65, label=model)
    lim = [0, max(1.0, enriched.overlap_der.max(skipna=True) or 1.0)]
    ax.plot(lim, lim, "k--", lw=1, label="equal difficulty")
    ax.set_xlabel("DER on single-speaker regions")
    ax.set_ylabel("DER on overlapped regions")
    ax.set_title("Points above the line = overlap is harder for that clip")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2.7 — Export the standalone error explorer

Writes a self-contained `error_explorer/` you download once and open locally.
No Colab, no backend, no build step, no internet.

Everything it shows was computed here and serialised — the browser does no metric
arithmetic. That is deliberate: a second implementation in JavaScript would be
free to drift from the one that produced the benchmark. The exported MISS / FA /
CONFUSION regions are **verified at export time** to reproduce pyannote's own DER
components to within a microsecond, and the result is recorded in
`data/clips.json` so the UI can warn you if it ever fails.

Audio is not bundled by default — the corpus is 1.3 GB and does not belong in a
git repo. The timeline, error regions and filtering all work without it; copy
individual WAVs into `error_explorer/audio/` for the clips you want to hear, or
pass `copy_audio=True`.

To use it:

```bash
cd error_explorer
python -m http.server 8000
# open http://localhost:8000
```

In [ ]:
from sarvam_diar import explorer

if not len(metrics):
    print("nothing to export -- run 2.1/2.2 first")
else:
    out = explorer.export(
        cfg, metrics, references, hypotheses,
        out_dir=cfg.root / "error_explorer",
        copy_audio=False,   # True bundles ~1.3 GB of WAV
        verify=True,        # assert regions reproduce the scored components
    )

    import json
    manifest = json.load(open(out / "data" / "clips.json"))
    bad = manifest["verification"]["mismatches"]
    print(f"\n{manifest['n_clips']} clips, models={manifest['models']}")
    print(f"verification mismatches: {len(bad)}"
          + ("  <-- regions would NOT match the metric, do not trust the view" if bad else "  (regions match the metric)"))

    size = sum(f.stat().st_size for f in out.rglob("*") if f.is_file())
    print(f"export size: {size/1e6:.1f} MB at {out}")
    print("\nDownload the folder, then:")
    print("    cd error_explorer && python -m http.server 8000")
    print("    open http://localhost:8000")

    # Optional: zip it so it downloads as one file from Colab.
    import shutil
    zip_path = shutil.make_archive(str(cfg.root / "error_explorer"), "zip", root_dir=out)
    print(f"\nzipped -> {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
    if IN_COLAB:
        from google.colab import files  # type: ignore[import-not-found]
        print("uncomment the next line to download it directly:")
        print("# files.download(zip_path)")

---

# Step 3 — ASR on diarized segments

Speaker-attributed transcripts from more than one STT system, scored with
**cpWER** and **WDER**, plus plain **WER** and **DI-cpWER** as diagnostics.

**Two strategies, and the difference between them is a result.**

* *Per-segment* — cut the audio at the diarized turn boundaries and transcribe
  each turn. Attribution is exact by construction. This is the literal reading
  of the brief, and for Sarvam it is the **only** option: Saaras returns exactly
  one timestamp span per request (measured, across v3/v4, 5 s/20 s, every mode),
  so there are no word times to attribute with.
* *Long-form* — transcribe the clip once with word timestamps and assign each
  word to the turn covering it. Only Whisper can do this, so running it on
  Whisper isolates the **strategy** effect rather than confounding it with the
  system.

Both systems therefore run per-segment for the like-for-like comparison, and
Whisper additionally runs long-form.

**Turns come from `reverb-v2`** — 2,650 merged segments at a median 7.66 s,
against community-1's 10,279 at 1.13 s. Fewer, longer segments give the
recogniser context instead of fragments. Adjacent same-speaker turns are merged
first; that is an ASR segmentation choice and the scored diarization is
untouched.

**No language is ever hinted.** Sarvam is called with `language_code="unknown"`
and Whisper with `language=None`. `ref_lang_script` is derived from the
ground-truth transcript, so passing it would be exactly the leak the brief
forbids. What each system detects is recorded and can be scored separately.


## 3.1 — Systems and inputs

In [ ]:
# faster-whisper, not whisperx: whisperx pins an older pyannote and would fight
# the Step 2 environment, while the only part of it we want -- assigning words
# to speakers -- is asr.assign_words(). faster-whisper is CTranslate2 underneath
# and pulls no pyannote at all, so it coexists with what is already installed.
%pip install -q faster-whisper rapidfuzz

from sarvam_diar import asr, text_metrics as tm

# This kernel keeps whatever it imported first, so a pulled fix does not reach
# a re-run cell until the modules are purged. Name the symbols this section
# needs and fail here rather than deep inside a library.
utils.assert_fresh("SARVAM_MAX_SEC", "merge_same_speaker", "run_segmented",
                   "oracle_ceiling", module=asr)
utils.assert_fresh("di_cpwer", "verify_assignment_exact", module=tm)

# SARVAM_API_KEY: .env at the working root, or Kaggle Secrets / Colab Secrets.
# Leave SARVAM off if you have no key -- Whisper alone still satisfies nothing,
# the brief wants more than one system, but the cells below will run.
try:
    _k = asr.resolve_sarvam_key(cfg)
    print(f"Sarvam key: {_k[:6]}...{_k[-4:]}")
    SARVAM_SYSTEMS = ["sarvam-saaras-v3", "sarvam-saaras-v4"]
except Exception as exc:
    print(f"no Sarvam key ({exc}); running Whisper only")
    SARVAM_SYSTEMS = []

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# large-v3 was measured at RTF ~0.5 on a T4 -- roughly 6.6 h for this corpus,
# which does not fit a session. large-v3-turbo has 4 decoder layers instead of
# 32 and decoding is where long-form time goes. It stays MULTILINGUAL, which the
# distil-* models are not -- they are English-only and so unusable across nine
# Indic scripts. Switch back to "whisper-large-v3" if there is time to spend.
WHISPER = "whisper-large-v3-turbo"

DIAR = "reverb-v2"          # which diarization the ASR is attributed against
import dataclasses
ready = [dataclasses.replace(ci, wav_path=str(cfg.wav_path(cid)))
         for cid, ci in inputs.items() if cfg.wav_path(cid).exists()]
have = sum(1 for c in ready if diarization.is_done(cfg, DIAR, c.clip_id))
print(f"{len(ready)} clips with audio; {have} have {DIAR} turns")
assert have, f"no {DIAR} hypotheses -- run Step 2 first"


## 3.2 — Smoke gate

In [ ]:
# --- SMOKE GATE ------------------------------------------------------------
# Whisper pads every input to a fixed 30 s window, so a 1 s segment costs the
# same encoder pass as a 30 s one: per-segment cost tracks the NUMBER of calls,
# not the audio length. Measure both paths on two clips before committing.
#
# Batched long-form is timed too, but it is NOT the default: batching requires
# Whisper's own Silero VAD, which is a second speech/non-speech decision on top
# of the diarizer's, and whatever it drops becomes an unrecoverable miss. Use it
# only if the unbatched projection does not fit the session, and say so in the
# writeup if you do.
import time

probe = sorted(ready, key=lambda c: c.duration)
probe = [probe[0], probe[len(probe) // 2]]      # not the longest: slowest, and
                                                # tells us the least
total_audio = sum(c.duration for c in ready)

for label, kwargs in (("long-form (greedy, no extra VAD)", dict(beam_size=1)),
                      ("long-form (batched, Whisper VAD)", dict(beam_size=1, batch_size=8))):
    print(label)
    rtfs = []
    for c in probe:
        t = time.time()
        words, meta = asr.transcribe_whisper(cfg, Path(c.wav_path),
                                             word_timestamps=True, **kwargs)
        el = time.time() - t
        rtfs.append(el / c.duration)
        print(f"  {c.clip_id:<26}{c.duration:7.0f}s  {el:7.1f}s  rtf {el/c.duration:.3f}"
              f"  {len(words):>5} words  {meta.get('detected_language')}")
    mean = sum(rtfs) / len(rtfs)
    print(f"  mean RTF {mean:.3f}  ->  {mean*total_audio/3600:.1f} h for all "
          f"{len(ready)} clips\n")

print("per-segment -- one call per turn, for the like-for-like comparison")
for system in ([WHISPER] + SARVAM_SYSTEMS[:1]):
    c = probe[0]
    turns = asr.merge_same_speaker(diarization.load_hypothesis(cfg, DIAR, c.clip_id), 1.0)
    t = time.time()
    segs, meta = asr.transcribe_segments(cfg, system, Path(c.wav_path), turns, workers=4)
    el = time.time() - t
    per = el / max(len(segs), 1)
    print(f"  {system:<24}{len(segs):>4} segs  {el:6.1f}s  ({per:.2f}s/seg)"
          f"  -> {per*2650/60:.0f} min for all 2,650 segments")


## 3.3 — Per-segment sweep

In [ ]:
# --- sweeps ----------------------------------------------------------------
# Whisper LONG-FORM over everything: 99 model calls rather than ~2,650, and it
# is the strategy Step 4 needs anyway, since reasoning about a transcript
# requires word times.
# Whisper PER-SEGMENT over a subset only: enough to measure the strategy gap
# without paying for it 99 times.
# Sarvam per-segment over everything, because Saaras returns no word timings and
# has no other option.
RUN_ASR = True
SEGMENT_SUBSET = 15        # clips for the Whisper per-segment comparison

if RUN_ASR:
    lf = asr.run(cfg, ready, flags=flags, systems=[WHISPER])
    if len(lf):
        display(lf[["clip_id", "n_words", "elapsed_sec", "rtf", "detected_language"]].head(8))

    if SARVAM_SYSTEMS:
        asr.run_segmented(cfg, ready, diar_model=DIAR, systems=SARVAM_SYSTEMS,
                          flags=flags, merge_gap=1.0, workers=4)

    subset = sorted(ready, key=lambda c: c.duration)[:SEGMENT_SUBSET]
    print(f"\nWhisper per-segment on {len(subset)} clips "
          f"({sum(c.duration for c in subset)/60:.0f} min of audio) for the "
          f"strategy comparison")
    asr.run_segmented(cfg, subset, diar_model=DIAR, systems=[WHISPER],
                      flags=flags, merge_gap=1.0)
else:
    print("RUN_ASR is False -- scoring whatever is already checkpointed")


## 3.4 — Long-form Whisper

In [ ]:
# Long-form already ran above, over every clip. Kept as its own step only to
# make the resume path obvious: re-running this is a no-op once checkpointed.
lf = asr.run(cfg, ready, flags=flags, systems=[WHISPER])
print(f"{len(lf)} long-form transcripts on disk")


## 3.5 — Score

In [ ]:
# --- score: WER, cpWER, DI-cpWER, WDER -------------------------------------
from sarvam_diar import reference as refmod

# Hungarian is EXACT for cpWER, not an approximation of a permutation search: a
# speaker-pair's error count does not depend on the other pairs, so the total is
# a sum of independent cells. Checked here rather than asserted.
_probe = [cid for cid in references][:12]
_checks = []
for cid in _probe:
    ref = references[cid]
    rt = {k: v.split() for k, v in refmod.speaker_texts(ref).items()}
    if not rt:
        continue
    keys = list(rt)
    hyp = {f"S{i}": rt[k] for i, k in enumerate(keys[1:] + keys[:1])}   # permuted
    _checks.append(tm.verify_assignment_exact(rt, hyp))
_ok = [c for c in _checks if c.get("checked")]
print(f"Hungarian vs brute force: {sum(c['agree'] for c in _ok)}/{len(_ok)} agree "
      f"(up to {max((c['n_permutations'] for c in _ok), default=0)} permutations)")
assert all(c["agree"] for c in _ok), "Hungarian disagrees with brute force"

rows = []
# Whisper per-segment covers only SEGMENT_SUBSET clips, so its rows are a
# subset by design; is_done() below simply skips the clips it never ran.
for system in [WHISPER] + SARVAM_SYSTEMS:
    tag = f"{system}@{DIAR}"
    for cid, ref in references.items():
        if not asr.is_done(cfg, tag, cid):
            continue
        pairs = asr.load_pairs(cfg, tag, cid)
        row = tm.score_transcript(
            {k: v.split() for k, v in refmod.speaker_texts(ref).items()},
            asr.speaker_texts_from_words(pairs),
            refmod.word_stream(ref), pairs)
        row.update(clip_id=cid, asr_system=system, diar_model=DIAR,
                   strategy="segment", script=ref.stats.get("lang_script"))
        rows.append(row)

# long-form Whisper, attributed by time overlap against the same turns
for cid, ref in references.items():
    if not asr.is_done(cfg, WHISPER, cid):
        continue
    if not diarization.is_done(cfg, DIAR, cid):
        continue
    words = asr.load_words(cfg, WHISPER, cid)
    turns = diarization.load_hypothesis(cfg, DIAR, cid)
    pairs = [(tok, spk)
             for w, spk in asr.assign_words(words, turns)
             for tok in refmod.normalize_text(w, strip_gloss=False).split()]
    row = tm.score_transcript(
        {k: v.split() for k, v in refmod.speaker_texts(ref).items()},
        asr.speaker_texts_from_words(pairs),
        refmod.word_stream(ref), pairs)
    row.update(clip_id=cid, asr_system=WHISPER, diar_model=DIAR,
               strategy="longform", script=ref.stats.get("lang_script"))
    rows.append(row)

asr_scores = pd.DataFrame(rows)
if len(asr_scores):
    asr_scores.to_csv(cfg.step3_metrics_csv, index=False)
    print(f"\nscored {len(asr_scores)} (system, strategy, clip) rows")
else:
    print("nothing scored yet -- run the sweep cells above")


## 3.6 — Results

In [ ]:
if not len(asr_scores):
    print("nothing to report yet")
else:
    print("=== CORPUS (pooled: sum errors / sum reference words) ===")
    out = []
    for (system, strategy), g in asr_scores.groupby(["asr_system", "strategy"]):
        out.append({"asr_system": system, "strategy": strategy,
                    **tm.summarise(g.to_dict("records"))})
    corpus = pd.DataFrame(out).sort_values("cpwer").reset_index(drop=True)
    display(corpus[["asr_system", "strategy", "n_clips", "wer", "cpwer",
                    "di_cpwer", "cp_minus_di_cp", "wder", "n_ref_words"]].round(4))

    # The floor. A cpWER has to be read against this, not against zero: with a
    # perfect ASR and a perfect diarization, overlapped speech still costs this
    # much, because one transcript cannot carry two simultaneous speakers.
    ceil = asr.oracle_ceiling(cfg, references)
    print(f"\nOVERLAP CEILING (oracle ASR + oracle diarization): "
          f"cpWER {ceil['cpwer']:.4f}  WDER {ceil['wder']:.4f}  "
          f"(exactly 0 on the {ceil['n_clips_no_overlap']} clips with no overlap)")
    best = corpus.iloc[0]
    print(f"best system is {best['cpwer']:.4f}, i.e. "
          f"{best['cpwer']-ceil['cpwer']:.4f} above the achievable floor")

    print("\n=== by script (pooled) ===")
    per = []
    for (system, strategy, script), g in asr_scores.groupby(
            ["asr_system", "strategy", "script"]):
        per.append({"asr_system": system, "strategy": strategy, "script": script,
                    **tm.summarise(g.to_dict("records"))})
    display(pd.DataFrame(per).pivot_table(
        index="script", columns=["asr_system", "strategy"], values="cpwer").round(3))
